In [0]:
from pyspark.sql import functions as F

# Load tables
fact_encounters = spark.table("medical_project.gold.fact_encounters")
fact_procedures = spark.table("medical_project.gold.fact_procedures")
dim_dates = spark.table("medical_project.gold.dim_date")

# encounter_base
encounter_base = (
    fact_encounters
    .join(dim_dates, F.to_date(fact_encounters["start"]) == dim_dates["date"], "left")
    .select(
        dim_dates["year"],
        dim_dates["quarter"],
        dim_dates["month"],
        dim_dates["half_year"],
        fact_encounters["patient_id"],
        fact_encounters["payer_id"],
        fact_encounters["encounter_id"],
        fact_encounters["encounter_class"],
        fact_encounters["total_cost"].alias("total_claim_cost"),
        fact_encounters["has_payer_coverage"],
        F.datediff(fact_encounters["stop"], fact_encounters["start"]).alias("encounter_duration_days")
    )
)

# procedure_agg
procedure_agg = (
    fact_procedures
    .join(dim_dates, fact_procedures["procedure_date"] == dim_dates["date"], "left")
    .groupBy(
        dim_dates["year"],
        dim_dates["quarter"],
        dim_dates["month"],
        fact_procedures["encounter_id"]
    )
    .agg(
        F.count("*").alias("procedure_count"),
        F.sum(fact_procedures["base_cost"]).alias("total_procedure_cost")
    )
)

# Final aggregation
datacube = (
    encounter_base
    .join(
        procedure_agg,
        (encounter_base["encounter_id"] == procedure_agg["encounter_id"]) &
        (encounter_base["year"] == procedure_agg["year"]) &
        (encounter_base["quarter"] == procedure_agg["quarter"]) &
        (encounter_base["month"] == procedure_agg["month"]),
        "left"
    )
    .groupBy(
        encounter_base["year"], 
        encounter_base["quarter"], 
        encounter_base["month"], 
        encounter_base["half_year"], 
        encounter_base["payer_id"], 
        encounter_base["encounter_class"]
    )
    .agg(
        F.countDistinct(encounter_base["encounter_id"]).alias("total_encounters"),
        F.countDistinct(encounter_base["patient_id"]).alias("unique_patients"),
        F.sum(encounter_base["total_claim_cost"]).alias("total_encounter_cost"),
        F.sum(F.coalesce(procedure_agg["procedure_count"], F.lit(0))).alias("total_procedures"),
        F.sum(F.coalesce(procedure_agg["total_procedure_cost"], F.lit(0))).alias("total_procedure_cost")
    )
    .orderBy(encounter_base["year"], encounter_base["quarter"], encounter_base["month"], encounter_base["encounter_class"])
)

display(datacube)

# Save table
datacube.write.mode("overwrite").saveAsTable("medical_project.gold.datacube")